In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import unicodedata
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv("../data/raw/medicaments_amm_senegal.csv")

df.head()

df.isna().sum()

df.drop_duplicates(inplace=True)

df.dropna(inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7363 entries, 0 to 7454
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Nom du Médicament      7363 non-null   object
 1   DCI                    7363 non-null   object
 2   Dosage                 7363 non-null   object
 3   Présentation           7363 non-null   object
 4   Forme Galénique        7363 non-null   object
 5   Voie d'administration  7363 non-null   object
 6   Laboratoire            7363 non-null   object
 7   Classe Thérapeutique   7363 non-null   object
dtypes: object(8)
memory usage: 517.7+ KB


In [6]:
def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFD', text)
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])
    return text.lower().strip()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].apply(normalize_text)

display(df.head())
print(f"Dimensions actuelles du dataset : {df.shape}")

,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Voie d'administration,Laboratoire,Classe Thérapeutique
0,5 df,chlorure de sodium,5 g,fl/100ml,solution pour perfusion,intraveineuse,aculife healthcare pvt,solution pour rehydratation
1,5-fluorouracile sandoz,5-fluorouracile,250 mg / 5 ml,fl/5ml,solution injectable,injectable,sandoz,anticancereux
2,5-fluorouracile sandoz,5-fluorouracile,1000 mg / 20 ml,fl/20ml,solution injectable,intraveineuse,sandoz,anticancereux
4,abacavir,abacavir,300 mg,b/60,comprime,orale,hetero,antiretroviral
5,abacavir / lamivudine,abacavir / lamivudine,600 mg / 300mg,b/60,comprime dispersible,orale,cipla,antiviral


Dimensions actuelles du dataset : (7363, 8)


In [8]:
df_encoded = df.copy()
label_encoders = {}

for col in df_encoded.columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

display(df_encoded.head())

,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Voie d'administration,Laboratoire,Classe Thérapeutique
0,0,883,1666,427,284,61,28,635
1,1,8,1287,530,263,36,1082,214
2,1,8,659,476,263,61,1082,214
4,2,11,1408,263,38,95,564,321
5,3,13,1974,263,50,95,312,358


In [10]:
# Vectorization des DCI pour permettre une recherche textuelle intelligente
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
tfidf_matrix = vectorizer.fit_transform(df['DCI'].fillna(''))

In [11]:
def rechercher_equivalents(requete, top_n=10):
    requete_normalisee = normalize_text(requete)

    requete_vec = vectorizer.transform([requete_normalisee])
    similarites = cosine_similarity(requete_vec, tfidf_matrix).flatten()

    indices_pertinent = similarites.argsort()[::-1]

    resultats = []
    for idx in indices_pertinent:
        if similarites[idx] > 0.1:
            resultats.append({
                'Nom du Médicament': df.iloc[idx]['Nom du Médicament'],
                'DCI': df.iloc[idx]['DCI'],
                'Dosage': df.iloc[idx]['Dosage'],
                'Présentation': df.iloc[idx]['Présentation'],
                'Forme Galénique': df.iloc[idx]['Forme Galénique'],
                'Laboratoire': df.iloc[idx]['Laboratoire'],
                'Classe Thérapeutique': df.iloc[idx]['Classe Thérapeutique'],
                'Score de Similarité': round(similarites[idx], 4)
            })
        if len(resultats) >= top_n:
            break

    df_res = pd.DataFrame(resultats)
    return df_res

In [12]:
# Exemple de recherche d'équivalents pour le paracétamol
resultats_recherche = rechercher_equivalents("paracetamol", top_n=5)
display(resultats_recherche)

,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Laboratoire,Classe Thérapeutique,Score de Similarité
0,halopara,paracetamol,1000mg,b/16,comprime effervescent,halewood laboratories pvt. ltd. 319,analgesique / antipyretique,1.0
1,halopara,paracetamol,1000 mg,b/8,comprime effervescent,halewood laboratories pvt. ltd. 319,antalgique / antipyretique,1.0
2,stopalgic,paracetamol,1000 mg,b/400,comprime,biocodex,—,1.0
3,stopalgic,paracetamol,1 g,b/400,comprime,medis,—,1.0
4,stopalgic,paracetamol,1 g,fl/100 ml,solution injectable,medis-s.a,—,1.0
